# ML Zoomcamp 2026 — Homework 5 (Deploying Machine Learning Models)
**Nicolau Calado Jofilsan** · 2026 cohort (DataTalks.Club)

**EN.** Homework 5 packages a frozen lead-scoring model (`DictVectorizer` + logistic regression, trained on
`course_lead_scoring_2026.csv`) behind a FastAPI service and a Docker container. The course provides the
artifact (`pipeline.bin`), the API (`predict.py`), the lockfile and the Dockerfile; the work is to verify the
artifact, reproduce the reference environment, score leads in three ways (loaded model, API, container) and read
the configuration. Deadline: Mon 26/10/2026, 23:00 (platform timezone).

**PT.** A atividade 5 coloca um modelo de lead scoring congelado (`DictVectorizer` + regressão logística,
treinado com `course_lead_scoring_2026.csv`) atrás de uma API FastAPI e de um contêiner Docker. O curso fornece o
artefato (`pipeline.bin`), a API (`predict.py`), o arquivo de travas e o Dockerfile; o trabalho é verificar o
artefato, reproduzir o ambiente de referência, pontuar leads de três formas (modelo carregado, API, contêiner) e
ler a configuração. Prazo: seg 26/10/2026, 23:00 (fuso da plataforma).

Reference environment / Ambiente de referência: Python 3.11.15, scikit-learn 1.7.2, numpy 2.3.3, FastAPI 0.119.1,
uvicorn 0.38.0, all pinned in `uv.lock`. Create it with / Crie com: `uv sync --locked`.

## 0. Course files and integrity check / Arquivos do curso e verificação de integridade

In [ ]:
import hashlib
import json
import pickle
import re
import subprocess
import sys
import tomllib
import urllib.request
import warnings
from pathlib import Path

BASE_URL = (
    "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/"
    "main/cohorts/2026/homework/05-deployment/"
)
FILES = [
    "pipeline.bin", "model.py", "predict.py", "feature_defaults.json", "model_metadata.json",
    "smoke_test.py", "q6_test.py", "pyproject.toml", "uv.lock", ".python-version", "Dockerfile",
]
EXPECTED_SHA256 = "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1"

WORKDIR = Path("course_files")
WORKDIR.mkdir(exist_ok=True)
for name in FILES:
    if not (WORKDIR / name).exists():
        urllib.request.urlretrieve(BASE_URL + name, WORKDIR / name)

print("downloaded / baixados:", sorted(p.name for p in WORKDIR.iterdir()))

**EN.** A pickle file can execute arbitrary code when loaded, so the artifact is checked *before* it is opened:
its SHA-256 must match the value published in the homework and in `model_metadata.json`.
**PT.** Um arquivo pickle pode executar código arbitrário ao ser carregado, então o artefato é verificado *antes*
de ser aberto: o SHA-256 precisa bater com o valor publicado no enunciado e em `model_metadata.json`.

In [ ]:
model_path = WORKDIR / "pipeline.bin"
metadata = json.loads((WORKDIR / "model_metadata.json").read_text(encoding="utf-8"))

digest = hashlib.sha256(model_path.read_bytes()).hexdigest()
print("sha256 of pipeline.bin:", digest)
print("matches the homework / bate com o enunciado:     ", digest == EXPECTED_SHA256)
print("matches model_metadata.json / bate com o metadata:", digest == metadata["model_sha256"])

defaults_digest = hashlib.sha256((WORKDIR / "feature_defaults.json").read_bytes()).hexdigest()
print("feature_defaults.json matches metadata / confere:  ", defaults_digest == metadata["feature_defaults_sha256"])

assert digest == EXPECTED_SHA256, "unexpected artifact, do not unpickle / artefato inesperado, não abrir"

## Q1. Environment check / Verificação do ambiente

**EN.** `uv --version` — a setup check, not graded (the local `uv` version says nothing about the model).
**PT.** `uv --version` — verificação de preparação, sem nota (a versão local do `uv` não é propriedade do modelo).

In [ ]:
try:
    print(subprocess.run(["uv", "--version"], capture_output=True, text=True, check=True).stdout.strip())
except (FileNotFoundError, subprocess.CalledProcessError):
    print("uv not found on PATH / uv não encontrado no PATH")

## Q2. Locked dependency / Dependência travada

**EN.** Which scikit-learn version belongs to the 2026 reference environment? Read it from both places the
homework names: the constraint in `pyproject.toml` and the resolved version in `uv.lock`.
**PT.** Qual versão do scikit-learn faz parte do ambiente de referência de 2026? Ler nos dois lugares que o
enunciado indica: a restrição em `pyproject.toml` e a versão resolvida em `uv.lock`.

In [ ]:
pyproject = tomllib.loads((WORKDIR / "pyproject.toml").read_text(encoding="utf-8"))
sk_constraint = [d for d in pyproject["project"]["dependencies"] if d.startswith("scikit-learn")][0]

lock = tomllib.loads((WORKDIR / "uv.lock").read_text(encoding="utf-8"))
sk_locked = [p["version"] for p in lock["package"] if p["name"] == "scikit-learn"][0]

print("pyproject.toml :", sk_constraint)
print("uv.lock        :", sk_locked)
print("\nQ2 =", sk_locked)

**Answer / Resposta Q2: `1.7.2`** — pinned as `scikit-learn==1.7.2` in `pyproject.toml` and resolved to `1.7.2`
in `uv.lock`; `model_metadata.json` records the same version for the training run.

EN: the lockfile, not the version installed on a given machine, defines the reference environment, which is why the
homework asks for `uv sync --locked` instead of regenerating it.
PT: o arquivo de travas, e não a versão instalada em cada máquina, define o ambiente de referência, por isso o
enunciado pede `uv sync --locked` em vez de regerá-lo.

## Q3. Load the model / Carregar o modelo

**EN.** Load `pipeline.bin` with `pickle` and call `predict_proba` for the lead in the homework. The pipeline
contains the `DictVectorizer`, so the lead goes in as a plain dictionary. `predict_proba` returns two columns,
`[P(not converted), P(converted)]`; the answer is column 1.
**PT.** Carregar `pipeline.bin` com `pickle` e chamar `predict_proba` para o lead do enunciado. O pipeline já
contém o `DictVectorizer`, então o lead entra como um dicionário comum. `predict_proba` devolve duas colunas,
`[P(não converteu), P(converteu)]`; a resposta é a coluna 1.

In [ ]:
import numpy as np
import sklearn

print("scikit-learn in this kernel / neste kernel:", sklearn.__version__, "| numpy:", np.__version__)
if sklearn.__version__ != sk_locked:
    print(f"WARNING: the model was saved with scikit-learn {sk_locked}; run with `uv sync --locked` for an exact reproduction.")

with warnings.catch_warnings():
    warnings.simplefilter("always")
    with model_path.open("rb") as model_file:
        pipeline = pickle.load(model_file)

print("pipeline steps / etapas:", [name for name, _ in pipeline.steps])

lead_q3 = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

probability_q3 = float(pipeline.predict_proba([lead_q3])[0, 1])
print("probability of conversion / probabilidade de conversão:", probability_q3)
print("\nQ3 =", f"{probability_q3:.3f}")

**Answer / Resposta Q3: 0.533** — 0.532994 before rounding (tolerance in the grader: 0.005).

EN: to check the model without trusting `predict_proba`, the next cell rebuilds the probability by hand: one-hot
vector times coefficients, plus intercept, through the sigmoid.
PT: para conferir o modelo sem depender do `predict_proba`, a próxima célula refaz a probabilidade à mão: vetor
one-hot vezes coeficientes, mais o intercepto, passando pela sigmoide.

In [ ]:
dv = pipeline.steps[0][1]
lr = pipeline.steps[-1][1]

x = dv.transform([lead_q3])
logit = float((x @ lr.coef_[0])[0] + lr.intercept_[0])
by_hand = 1 / (1 + np.exp(-logit))

print("features after one-hot / variáveis após one-hot:", len(dv.feature_names_))
print("logit:", round(logit, 4), "| sigmoid(logit):", round(float(by_hand), 6))
print("equal to predict_proba / igual ao predict_proba:", bool(np.isclose(by_hand, probability_q3)))

## Q4. Serve the model / Servir o modelo

**EN.** The course API (`predict.py`) loads the same pipeline, fills missing values with the training-set medians
(`feature_defaults.json`), and exposes `GET /health` and `POST /predict`. Start it from the course folder with the
locked environment:

```
cd course_files
uv sync --locked
uv run uvicorn predict:app --host 0.0.0.0 --port 9696
```

**PT.** A API do curso (`predict.py`) carrega o mesmo pipeline, preenche valores ausentes com as medianas do
treino (`feature_defaults.json`) e expõe `GET /health` e `POST /predict`. Inicie-a a partir da pasta do curso com o
ambiente travado (comandos acima), e então execute a célula abaixo em outro terminal ou depois de subir o servidor.

In [ ]:
import urllib.error
import urllib.request

API_URL = "http://localhost:9696"


def call_api(path, payload=None):
    """GET when payload is None, POST (JSON) otherwise."""
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(
        API_URL + path, data=data, headers={"Content-Type": "application/json"},
        method="GET" if payload is None else "POST",
    )
    with urllib.request.urlopen(request, timeout=10) as response:
        return json.load(response)


lead_q4 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}

try:
    print("GET /health ->", call_api("/health"))
    result_q4 = call_api("/predict", lead_q4)
    print("POST /predict ->", result_q4)
    assert 0 <= result_q4["conversion_probability"] <= 1 and isinstance(result_q4["conversion"], bool)
    print("\nQ4 =", f"{result_q4['conversion_probability']:.3f}")
except urllib.error.URLError:
    print("API not reachable on port 9696 / API não acessível na porta 9696: start it with the command above.")

**Answer / Resposta Q4: 0.770** — the API returns `conversion_probability` = 0.769799 and `conversion` = `true`.

EN: the same lead scored directly with the loaded pipeline (cell below) gives the same number, which confirms that
the API adds no transformation beyond the documented missing-value policy.
PT: o mesmo lead pontuado direto no pipeline carregado (célula abaixo) dá o mesmo número, o que confirma que a API
não acrescenta nenhuma transformação além da política documentada de valores ausentes.

In [ ]:
direct_q4 = float(pipeline.predict_proba([lead_q4])[0, 1])
print("direct / direto:", round(direct_q4, 6))
print("rounded to 3 decimals / arredondado em 3 casas:", f"{direct_q4:.3f}")

## Q5. Container configuration / Configuração do contêiner

**EN.** Which Python base-image tag does the Dockerfile declare? Read the `FROM` line (the digest after `@` pins
the exact image and is not part of the tag).
**PT.** Qual tag de imagem base do Python o Dockerfile declara? Ler a linha `FROM` (o digest depois do `@` fixa a
imagem exata e não faz parte da tag).

In [ ]:
dockerfile = (WORKDIR / "Dockerfile").read_text(encoding="utf-8")
from_line = [line for line in dockerfile.splitlines() if line.startswith("FROM ")][0]
image_with_digest = from_line.split()[1]
image_tag = image_with_digest.split("@")[0]

print("FROM line :", from_line)
print("\nQ5 =", image_tag)

**Answer / Resposta Q5: `python:3.11.15-slim-bookworm`** — it matches the interpreter in `.python-version` and
`requires-python = ">=3.11,<3.12"`, so the container runs the same Python as the local reference environment.

EN: pinning the image by digest (and `uv` by digest in the next `COPY --from` line) makes the build reproducible
even if the tag is later republished.
PT: fixar a imagem por digest (e o `uv` por digest na linha `COPY --from` seguinte) torna o build reprodutível,
mesmo que a tag seja republicada depois.

In [ ]:
print(".python-version :", (WORKDIR / ".python-version").read_text().strip())
print("requires-python :", pyproject["project"]["requires-python"])

## Q6. Run the container / Executar o contêiner

**EN.** Build and run the image, then send the same request as in Q4 to the container (port 9696) and check
`/health`. Stop any local server first so the port is free.

```
cd course_files
docker build -t zoomcamp-model:2026-hw5 .
docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5
```

**PT.** Construa e execute a imagem, depois envie ao contêiner (porta 9696) a mesma requisição da Q4 e confira
`/health`. Pare qualquer servidor local antes, para a porta ficar livre.

In [ ]:
try:
    health = call_api("/health")
    print("GET /health ->", health)
    print("model checksum matches model_metadata.json / checksum do modelo confere:",
          health["model_sha256"] == metadata["model_sha256"])

    result_q6 = call_api("/predict", lead_q4)
    print("POST /predict ->", result_q6)
    print("\nQ6 =", f"{result_q6['conversion_probability']:.3f}")
except urllib.error.URLError:
    print("Container not reachable on port 9696 / contêiner não acessível na porta 9696.")

**Answer / Resposta Q6: 0.770** — expected to equal Q4: the container runs the same code, lockfile and artifact,
so the same lead yields the same probability (grader tolerance: 0.005).

EN: `/health` should report `status: ok` and a `model_sha256` equal to the checksum recorded in
`model_metadata.json`, which ties the running container back to the artifact verified in section 0.
PT: `/health` deve informar `status: ok` e um `model_sha256` igual ao checksum registrado em
`model_metadata.json`, o que liga o contêiner em execução ao artefato verificado na seção 0.

### Homework 5 — answers / respostas

| Question | Answer |
|---|---|
| Q1 `uv 0.13.1` uv 0.13.1 (62faed335 2026-10-13 x86_64-pc-windows-msvc) | version installed locally / versão instalada localmente |
| Q2 scikit-learn in the reference environment | **`1.7.2`** |
| Q3 probability, loaded model (`paid_ads` lead) | **0.533** |
| Q4 probability, API (`organic_search` lead) | **0.770** |
| Q5 Python base image in the Dockerfile | **`python:3.11.15-slim-bookworm`** |
| Q6 probability, container (same request as Q4) | **0.770** |

### Notes / Observações

**EN.** 1) The artifact is verified by checksum before `pickle.load`, because unpickling untrusted files can run
code. 2) The model was trained with scikit-learn 1.7.2; loading it with another version can emit a warning and, in
principle, change results, which is why the environment is reproduced from `uv.lock`. 3) Missing numeric fields are
filled with training-set medians (`feature_defaults.json`), so an empty request still returns a valid probability;
that policy lives in `model.py`, shared by the script, the API and the container.

**PT.** 1) O artefato é verificado por checksum antes do `pickle.load`, porque abrir arquivos não confiáveis pode
executar código. 2) O modelo foi treinado com scikit-learn 1.7.2; carregá-lo com outra versão pode gerar aviso e,
em princípio, alterar resultados, por isso o ambiente é reproduzido a partir do `uv.lock`. 3) Campos numéricos
ausentes são preenchidos com as medianas do treino (`feature_defaults.json`), então uma requisição vazia ainda
devolve uma probabilidade válida; essa política está em `model.py`, compartilhado pelo script, pela API e pelo
contêiner.